# Fonte Gaussian pulse

A fonte usada no FDTD é um **pulso Gaussian** modulado em seno:

$$s(t) = \exp\left(-\frac{(t-t_0)^2}{2\tau^2}\right) \sin(\omega_0(t-t_0))$$

Com $\omega_0 = 2\pi f_0$, $\tau = 1/(2\pi \cdot \mathrm{bandwidth})$ e $t_0 = 4.5\tau$ para centrar o pulso. Abaixo: comparamos a forma **teórica** com o campo **injectado** na simulação no ponto de injeção.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

ROOT = Path.cwd()
for _ in range(5):
    if (ROOT / "emsim").is_dir():
        break
    if ROOT == ROOT.parent:
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from emsim.fdtd.grid import YeeGrid
from emsim.fdtd.fields import update_E, update_H
from emsim.sources.gaussian_pulse import GaussianPulse

In [ ]:
f0 = 10e9
bandwidth = 4e9
source = GaussianPulse(f0=f0, bandwidth=bandwidth)

t_theory = np.linspace(0, 2e-9, 500)
s_theory = np.array([float(source(t).numpy()) for t in t_theory])

In [ ]:
grid = YeeGrid(
    x_range=(0, 5e-3), y_range=(0, 5e-3), z_range=(0, 20e-3),
    f0=f0, resolution=30, courant=0.5,
)
z_src = grid.Nz // 2
jy, jx = grid.Ny // 2, grid.Nx // 2
mat = grid.materials
coeffs = grid.get_curl_coefficients()
inv_dx, inv_dy, inv_dz = coeffs["inv_dx"], coeffs["inv_dy"], coeffs["inv_dz"]

Ey_inj = []
n_steps = 400
for n in range(n_steps):
    update_H(grid.Ex, grid.Ey, grid.Ez, grid.Hx, grid.Hy, grid.Hz,
             mat.dt_over_mu, inv_dx, inv_dy, inv_dz)
    update_E(grid.Ex, grid.Ey, grid.Ez, grid.Hx, grid.Hy, grid.Hz,
             mat.Ca, mat.Cb, inv_dx, inv_dy, inv_dz)
    amp = float(source(n * grid.dt).numpy())
    idx = tf.constant([[z_src, jy, jx]], dtype=tf.int32)
    new_val = grid.Ey[z_src, jy, jx].numpy() + amp
    grid.Ey.assign(tf.tensor_scatter_nd_update(
        grid.Ey.read_value(), idx, tf.constant([new_val], dtype=grid.Ey.dtype)))
    Ey_inj.append(grid.Ey[z_src, jy, jx].numpy())

Ey_inj = np.array(Ey_inj)
t_sim = np.arange(n_steps) * grid.dt

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(9, 4))
ax.plot(t_theory * 1e9, s_theory, label="Teoria: s(t)", linestyle="--")
ax.plot(t_sim * 1e9, Ey_inj, label="Simulacao: Ey no ponto de injecao")
ax.set_xlabel("Tempo [ns]")
ax.set_ylabel("Amplitude")
ax.legend()
ax.set_title("Forma do pulso: teoria vs campo injectado na malha")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()